In [0]:
# Define table variables from dbutils widgets
billinginvoices_history = dbutils.widgets.get("billinginvoices_history")
branch = dbutils.widgets.get("branch")
office = dbutils.widgets.get("office")
clientepisodefsall = dbutils.widgets.get("clientepisodefsall")
clientepisodesall = dbutils.widgets.get("clientepisodesall")
payortype = dbutils.widgets.get("payortype")
payorsources = dbutils.widgets.get("payorsources")
client = dbutils.widgets.get("client")
accountingcashtransactions_history = dbutils.widgets.get("accountingcashtransactions_history")
billingsubmissioninvoices = dbutils.widgets.get("billingsubmissioninvoices")
billingsubmissions = dbutils.widgets.get("billingsubmissions")
clientepisodevisitsall = dbutils.widgets.get("clientepisodevisitsall")
target_table = dbutils.widgets.get("target_table")

In [0]:
spark.sql(f"""
TRUNCATE TABLE {target_table}
""")

In [0]:
display(
spark.sql(f"""
INSERT INTO {target_table}
SELECT
    bi_h.i_id AS InvNum,
    CONCAT(c.LastName, ', ', c.FirstName) AS ClientName,
    DATE(c.DateOfBirth) AS DOB,
    'BAYADA HOME HEALTH CARE, INC.' AS HCHB_Company,
    bi_h.i_balance AS Client_Balance,
    date_format(epi.epi_StartOfEpisode, 'yyyy-MM-dd') AS EpisodeStart,
    date_format(epi.epi_DischargeDate, 'yyyy-MM-dd') AS EpisodeEnd,
    COALESCE(
        NULLIF(TRIM(REPLACE(REPLACE(REPLACE(cefs.cefs_policyno, CHAR(13), ''), CHAR(10), ''), CHAR(9), '')), ''),
        cefs.cefs_medicareno
    ) AS Subscriber_Id,
    NULL AS MedicareBeneficiaryIdentifier,
    NULL AS BillingNoteCount,
    date_format(sub.FirstSubmissionDate, 'yyyy-MM-dd') AS FirstSubmissionDate,
    date_format(pmt.LastPaymentApplied, 'MM/dd/yyyy') AS LastPaymentApplied,
    date_format(visits.FirstDOS, 'MM/dd/yyyy') AS FirstDOS,
    date_format(visits.LastDOS, 'MM/dd/yyyy') AS LastDOS,
    bi_h.i_charge AS Charges,
    pmt.TotalPayments AS Payments,
    bi_h.i_balance AS StillOwe,
    CAST(0 AS DECIMAL(10,4)) AS 0_60,
    CAST(0 AS DECIMAL(10,4)) AS 61_90,
    CAST(0 AS DECIMAL(10,4)) AS 91_120,
    CAST(0 AS DECIMAL(10,4)) AS 121_150,
    CAST(0 AS DECIMAL(10,4)) AS 151_180,
    CAST(0 AS DECIMAL(10,4)) AS 181_,
    'No Notes' AS LastBillingNote,
    'No Note' AS LastBillingNoteDate,
    'No Note' AS LastBillingNoteType,
    'No Note' AS LastBillingNoteCommentType,
    NULL AS LastFollowupDate,
    'No Note' AS LastBillingNoteUser,
    pt.PayorType AS PayorType,
    ps.ps_desc AS PayorName,
    NULL AS PayorClaimNumber,
    CONCAT('OFC: ', ofc.OfficeAbbreviation, '/', ofc.OfficeNumber, ' - ', UPPER(SPLIT(ofc.OfficeName, ' ')[SIZE(SPLIT(ofc.OfficeName, ' ')) - 1])) AS BranchName,
    ofc.OfficeNumber AS BranchNum,  -- FIXED
    b.branch_code AS BranchID,  -- FIXED
    ofc.Division AS Division,
    NULLIF(ofc.Area, '[Area not assigned]') AS Area,
    ofc.Region AS Region,
    c.State AS State,
    epi.epi_id AS EpisodeID,
    CASE ps.ps_freq 
        WHEN 0 THEN 'On Demand'
        WHEN 6 THEN 'Episodic'
        ELSE 'On Demand'
    END AS BillingFrequency,
    CAST(0 AS DECIMAL(10,4)) AS 181_270,
    CAST(0 AS DECIMAL(10,4)) AS 271_360,
    CAST(0 AS DECIMAL(10,4)) AS 361_450,
    CAST(0 AS DECIMAL(10,4)) AS 451_540,
    CAST(0 AS DECIMAL(10,4)) AS 541_,
    date_format(bi_h.i_postdate, 'yyyy-MM-dd') AS BillDate,
    NULL AS PdgmPeriodId,
    date_format(bi_h.i_insertdate, 'yyyy-MM-dd HH:mm:ss') AS DateCreated,
    current_timestamp() AS landing_update_datetime,
    NULL AS landing_update_by
FROM {billinginvoices_history} bi_h
JOIN {branch} b  
    ON bi_h.i_branchcode = b.branch_code 
LEFT JOIN {office} ofc 
    ON ofc.OfficeNumber = b.branch_code
JOIN {clientepisodefsall} cefs
    ON cefs.cefs_id = bi_h.i_cefsid
JOIN {clientepisodesall} epi
    ON epi.epi_id = cefs.cefs_epiid
LEFT JOIN {payortype} pt
    ON pt.pt_id = cefs.cefs_ptid
LEFT JOIN {payorsources} ps
    ON ps.ps_id = cefs.cefs_psid
JOIN (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY pa_id ORDER BY ClientID DESC) as rn
    FROM {client}
    WHERE Deleted = false
) c
    ON c.pa_id = epi.epi_paid
    AND c.rn = 1
LEFT JOIN (
    SELECT 
        ct_iid,
        SUM(ct_initialamount) AS TotalPayments,
        MAX(ct_postdate) AS LastPaymentApplied
    FROM {accountingcashtransactions_history}
    WHERE ct_cttid = 2
    GROUP BY ct_iid
) pmt
    ON pmt.ct_iid = bi_h.i_id
LEFT JOIN (
    SELECT 
        bsi.bsi_invnum,
        MIN(bs.bs_createdate) AS FirstSubmissionDate
    FROM {billingsubmissioninvoices} bsi
    JOIN {billingsubmissions} bs
        ON bs.bs_id = bsi.bsi_bsid
    GROUP BY bsi.bsi_invnum
) sub
    ON sub.bsi_invnum = bi_h.i_id
LEFT JOIN (
    SELECT 
        cefs.cefs_id,
        MIN(v.CEV_VISITDATE) as FirstDOS,
        MAX(v.CEV_VISITDATE) as LastDOS
    FROM {clientepisodefsall} cefs
    LEFT JOIN {clientepisodevisitsall} v
        ON v.CEV_EPIID = cefs.cefs_epiid
    GROUP BY cefs.cefs_id
) visits
    ON visits.cefs_id = cefs.cefs_id
""")
)